<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/01_signal_construction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Signal Construction

This notebook builds the project's first alternative-data signal: a **congressional
cluster-buy score**, and the **enrichment layer** that makes its smartest features
real. The research question it serves is whether clustered, conviction-weighted
congressional buying identifies equities with informational edge ahead of the
broader market.

The signal is the worked template; the remaining signals (government contracts,
lobbying, off-exchange) follow the same `BaseSignal` contract and join a weighted
composite. Notebook `02` tests whether the scores predict forward returns and
calibrates the feature weights against that evidence.

### Signal design

The score blends features, each encoding a testable hypothesis about what makes a
disclosed purchase informative:

| Feature | Hypothesis | Data source |
|---|---|---|
| `cluster` | More distinct members buying the same name carries more signal | Quiver |
| `size_vs_networth` | Larger trades signal conviction — measured *relative to the member's net worth* to remove the wealth confound | Quiver + net-worth reference |
| `committee_alignment` | A member whose committee oversees the traded sector may be better informed | committee reference |
| `recency` | More recent disclosures are more actionable | Quiver |
| `bipartisan` | Agreement across parties strengthens the read | Quiver |

Two of these features draw on data the Quiver client does not return — net worth
and committee membership — so the project sources them itself in an **enrichment
layer** (built below). A discipline runs through the whole signal: scores are keyed
on the **disclosure date**, never the trade date, since a trade becomes actionable
only once it is public.


## Data and setup

The pipeline defaults to **mock trades**, so this notebook runs with no Quiver key.
To use live Quiver data, store your token in Colab Secrets as `QUIVER_API_KEY` and
set `USE_LIVE = True`. The enrichment layer pulls public committee data either way,
so its real features are demonstrated even in mock mode.


In [2]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"   # <-- edit this
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True once your token is saved in Secrets

Working in: /content/quant-equity-research


## Write the package modules

The signal layer and a new enrichment layer go into the installable package. The
enrichment layer carries a mock implementation (synthetic, offline) and a live one
that resolves real public data: committee membership from the
`@unitedstates/congress-legislators` project, net worth from a curated reference
table, and ticker sectors from a static map with a yfinance fallback.


In [3]:
os.makedirs("src/quant_research/enrichment", exist_ok=True)
os.makedirs("reference_data", exist_ok=True)
open("src/quant_research/enrichment/__init__.py", "w").write('"""Member and security enrichment used by the signals.\n\nTwo implementations behind one interface:\n  MockEnrichment  - synthetic, offline, deterministic (for mock-data runs)\n  LiveEnrichment  - real public sources (committees, net worth, sectors)\n"""\n')
open("src/quant_research/enrichment/mock.py", "w").write('"""Synthetic enrichment: self-contained, offline, deterministic.\n\nUsed when running on mock trades without reaching any external source. The maps\nmirror the shape of the live data so signal behavior is comparable across modes.\n"""\n\nTICKER_SECTOR = {\n    "PLTR": "Technology", "NVDA": "Technology", "AAPL": "Technology", "MSFT": "Technology",\n    "LMT": "Defense", "RTX": "Defense", "NOC": "Defense", "AXON": "Defense", "BA": "Defense",\n    "CELH": "Consumer Staples", "MELI": "Consumer Discretionary", "CAVA": "Consumer Discretionary",\n    "GE": "Industrials", "CAT": "Industrials", "JPM": "Financials",\n}\nCOMMITTEE_SECTORS = {\n    "Armed Services": {"Defense"},\n    "Science, Space, and Technology": {"Technology"},\n    "Financial Services": {"Financials"},\n    "Energy and Commerce": {"Energy", "Technology"},\n    "Agriculture": {"Consumer Staples"},\n    "Transportation and Infrastructure": {"Industrials"},\n}\nREP_COMMITTEE = {\n    "Nancy Pelosi": "Science, Space, and Technology", "Dan Crenshaw": "Armed Services",\n    "Josh Gottheimer": "Financial Services", "Marjorie Taylor Greene": "Armed Services",\n    "Ro Khanna": "Armed Services", "Tommy Tuberville": "Armed Services",\n    "Michael McCaul": "Armed Services", "Virginia Foxx": "Science, Space, and Technology",\n    "Earl Blumenauer": "Transportation and Infrastructure", "Kathy Manning": "Financial Services",\n}\nREP_NETWORTH = {\n    "Nancy Pelosi": 250_000_000, "Dan Crenshaw": 3_000_000, "Josh Gottheimer": 8_000_000,\n    "Marjorie Taylor Greene": 11_000_000, "Ro Khanna": 60_000_000, "Tommy Tuberville": 6_000_000,\n    "Michael McCaul": 125_000_000, "Virginia Foxx": 9_000_000, "Earl Blumenauer": 2_000_000,\n    "Kathy Manning": 40_000_000,\n}\n\nSECTOR_REPS = {}\nfor _rep, _comm in REP_COMMITTEE.items():\n    for _sec in COMMITTEE_SECTORS.get(_comm, set()):\n        SECTOR_REPS.setdefault(_sec, []).append(_rep)\n\n\nclass MockEnrichment:\n    def net_worth(self, representative):\n        return REP_NETWORTH.get(representative)\n\n    def committee_sectors(self, representative):\n        return COMMITTEE_SECTORS.get(REP_COMMITTEE.get(representative), set())\n\n    def sector_of(self, ticker):\n        return TICKER_SECTOR.get(ticker)\n\n    def is_aligned(self, representative, ticker):\n        sec = self.sector_of(ticker)\n        return sec is not None and sec in self.committee_sectors(representative)\n')
open("src/quant_research/enrichment/live.py", "w").write('"""Live enrichment from public sources.\n\n- Committee membership: the @unitedstates/congress-legislators project, fetched and\n  cached locally. Members are matched from a Quiver representative name to a bioguide\n  id, then to their current committee assignments.\n- Net worth: a curated reference table (reference_data/networth.csv), with a small\n  embedded fallback. Figures are approximate public estimates and are meant to be\n  expanded from primary financial-disclosure data over time.\n- Ticker sector: a static map with an optional yfinance fallback.\n\nUnmatched members and unmapped tickers degrade silently — the feature simply does\nnot fire, which never penalizes a name, it only declines to boost it.\n"""\nimport os\nimport re\nimport csv\nimport json\nimport unicodedata\n\nUS_BASE = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/"\nDEFAULT_CACHE = "data/reference"\nNETWORTH_CSV = "reference_data/networth.csv"\n\n# committee-name keyword -> sectors it has oversight relevance to\nSECTOR_RULES = [\n    ("armed services", {"Defense"}),\n    ("homeland security", {"Defense"}),\n    ("foreign affairs", {"Defense"}),\n    ("financial services", {"Financials"}),\n    ("banking", {"Financials"}),\n    ("ways and means", {"Financials"}),\n    ("energy", {"Energy", "Utilities"}),\n    ("natural resources", {"Energy"}),\n    ("commerce", {"Technology", "Consumer Discretionary"}),\n    ("science", {"Technology"}),\n    ("agriculture", {"Consumer Staples"}),\n    ("health", {"Health Care"}),\n    ("transportation", {"Industrials"}),\n    ("infrastructure", {"Industrials"}),\n]\n\n# static ticker -> sector; extend freely. Sectors match SECTOR_RULES outputs.\nTICKER_SECTOR = {\n    "PLTR": "Technology", "NVDA": "Technology", "AAPL": "Technology", "MSFT": "Technology",\n    "GOOGL": "Technology", "AMZN": "Consumer Discretionary", "META": "Technology",\n    "LMT": "Defense", "RTX": "Defense", "NOC": "Defense", "AXON": "Defense",\n    "BA": "Defense", "GD": "Defense", "LHX": "Defense",\n    "XOM": "Energy", "CVX": "Energy", "COP": "Energy", "NEE": "Utilities", "DUK": "Utilities",\n    "JPM": "Financials", "BAC": "Financials", "GS": "Financials", "MS": "Financials",\n    "UNH": "Health Care", "JNJ": "Health Care", "PFE": "Health Care", "LLY": "Health Care",\n    "CELH": "Consumer Staples", "KO": "Consumer Staples", "PG": "Consumer Staples",\n    "MELI": "Consumer Discretionary", "CAVA": "Consumer Discretionary", "NKE": "Consumer Discretionary",\n    "GE": "Industrials", "CAT": "Industrials", "HON": "Industrials", "DE": "Industrials",\n}\n\n# yfinance sector strings -> our sector labels\n_YF_SECTOR = {\n    "Technology": "Technology", "Industrials": "Industrials",\n    "Financial Services": "Financials", "Healthcare": "Health Care",\n    "Energy": "Energy", "Utilities": "Utilities",\n    "Consumer Defensive": "Consumer Staples", "Consumer Cyclical": "Consumer Discretionary",\n}\n\n# approximate public net-worth estimates (USD), keyed by bioguide; clearly rough,\n# meant as a fallback when reference_data/networth.csv is absent. Expand from\n# primary financial-disclosure data for production use.\n_NETWORTH_FALLBACK = {\n    "P000197": 120_000_000,   # Pelosi\n    "T000278": 6_000_000,     # Tuberville\n    "K000389": 30_000_000,    # Khanna\n    "M001157": 100_000_000,   # McCaul\n    "C001120": 3_000_000,     # Crenshaw\n}\n\n_SUFFIX = {"jr", "sr", "ii", "iii", "iv", "v"}\n\n\ndef _norm(s):\n    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()\n    return re.sub(r"\\s+", " ", re.sub(r"[^a-z ]", " ", s.lower())).strip()\n\n\ndef _firstlast(s):\n    toks = [t for t in _norm(s).split() if t not in _SUFFIX]\n    return f"{toks[0]} {toks[-1]}" if len(toks) >= 2 else _norm(s)\n\n\ndef _committee_sectors(committee_name):\n    out, low = set(), committee_name.lower()\n    for kw, secs in SECTOR_RULES:\n        if kw in low:\n            out |= secs\n    return out\n\n\nclass LiveEnrichment:\n    def __init__(self, cache_dir=DEFAULT_CACHE, networth_csv=NETWORTH_CSV):\n        self.cache_dir = cache_dir\n        self.networth_csv = networth_csv\n        self._name_to_bio = {}\n        self._bio_committees = {}\n        self._networth = {}\n        self._load()\n\n    # ---- data loading ----------------------------------------------------\n    def _fetch_yaml(self, filename):\n        """Fetch a congress-legislators YAML, caching the parsed JSON locally."""\n        import requests\n        import yaml\n        os.makedirs(self.cache_dir, exist_ok=True)\n        cache = os.path.join(self.cache_dir, filename.replace(".yaml", ".json"))\n        if os.path.exists(cache):\n            with open(cache) as f:\n                return json.load(f)\n        data = yaml.safe_load(requests.get(US_BASE + filename, timeout=120).text)\n        with open(cache, "w") as f:\n            json.dump(data, f)\n        return data\n\n    def _load(self):\n        legislators = self._fetch_yaml("legislators-current.yaml")\n        committees = self._fetch_yaml("committees-current.yaml")\n        membership = self._fetch_yaml("committee-membership-current.yaml")\n\n        for p in legislators:\n            n, bio = p["name"], p["id"]["bioguide"]\n            variants = {f"{n.get(\'first\',\'\')} {n.get(\'last\',\'\')}", n.get("official_full", "")}\n            if n.get("nickname"):\n                variants.add(f"{n[\'nickname\']} {n.get(\'last\',\'\')}")\n            for v in variants:\n                if v.strip():\n                    self._name_to_bio[_norm(v)] = bio\n                    self._name_to_bio[_firstlast(v)] = bio\n\n        comm_name = {c["thomas_id"]: c["name"] for c in committees}\n        for cid, members in membership.items():\n            if len(cid) == 4:  # top-level committees, not subcommittees\n                name = comm_name.get(cid, cid)\n                for m in members:\n                    self._bio_committees.setdefault(m["bioguide"], set()).add(name)\n\n        self._networth = self._load_networth()\n\n    def _load_networth(self):\n        if os.path.exists(self.networth_csv):\n            out = {}\n            with open(self.networth_csv) as f:\n                for row in csv.DictReader(f):\n                    bio = row.get("bioguide", "").strip()\n                    val = row.get("net_worth", "").strip()\n                    if bio and val:\n                        out[bio] = float(val)\n            if out:\n                return out\n        return dict(_NETWORTH_FALLBACK)\n\n    # ---- lookups ---------------------------------------------------------\n    def resolve(self, representative):\n        return (self._name_to_bio.get(_norm(representative))\n                or self._name_to_bio.get(_firstlast(representative)))\n\n    def net_worth(self, representative):\n        return self._networth.get(self.resolve(representative))\n\n    def committee_sectors(self, representative):\n        comms = self._bio_committees.get(self.resolve(representative), set())\n        out = set()\n        for c in comms:\n            out |= _committee_sectors(c)\n        return out\n\n    def sector_of(self, ticker):\n        if ticker in TICKER_SECTOR:\n            return TICKER_SECTOR[ticker]\n        try:\n            import yfinance as yf\n            return _YF_SECTOR.get(yf.Ticker(ticker).info.get("sector"))\n        except Exception:\n            return None\n\n    def is_aligned(self, representative, ticker):\n        sec = self.sector_of(ticker)\n        return sec is not None and sec in self.committee_sectors(representative)\n')
print("wrote enrichment layer")

wrote enrichment layer


In [4]:
open("src/quant_research/ingestion/mock_data.py", "w").write('"""Synthetic data shaped like the Quiver Quantitative API responses.\n\nColumn names and value formats mirror the live `quiverquant` package so the same\nnormalization works on mock and real data. A few tickers carry heavy, purchase-\nskewed activity, and some buys are routed to representatives whose committee aligns\nwith the traded sector, so the signal\'s features have structure to detect. Member\nnames are real, so the live enrichment layer resolves them directly.\n"""\nimport numpy as np\nimport pandas as pd\nfrom datetime import date, timedelta\n\nfrom ..enrichment import mock as menr\n\n_UNIVERSE = ["PLTR", "LMT", "RTX", "AXON", "NOC", "CELH", "MELI", "CAVA",\n             "NVDA", "AAPL", "MSFT", "GE", "BA", "CAT", "JPM"]\n_HOT = ["PLTR", "LMT", "AXON"]\n_RANGES = ["$1,001 - $15,000", "$15,001 - $50,000", "$50,001 - $100,000",\n           "$100,001 - $250,000", "$250,001 - $500,000", "$500,001 - $1,000,000"]\n_REPS = list(menr.REP_COMMITTEE.keys())\n_TITLES = ["CEO", "CFO", "Director", "President", "EVP", "VP Finance", "COO"]\n_AGENCIES = ["Department of Defense", "Department of Energy", "NASA",\n             "Department of Homeland Security", "Department of Health"]\n_ISSUES = ["Defense", "Healthcare", "Taxation", "Energy", "Technology", "Trade"]\n\n\ndef _recent(rng, max_days_ago, min_days_ago=0):\n    return date.today() - timedelta(days=int(rng.integers(min_days_ago, max_days_ago)))\n\n\ndef mock_congress_trading(seed=42, n=180):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        if rng.random() < 0.45:\n            ticker = rng.choice(_HOT)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.85, 0.15])\n        else:\n            ticker = rng.choice(_UNIVERSE)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.55, 0.45])\n\n        sector = menr.TICKER_SECTOR.get(ticker)\n        aligned = menr.SECTOR_REPS.get(sector, [])\n        rep = rng.choice(aligned) if (aligned and rng.random() < 0.5) else rng.choice(_REPS)\n\n        report = _recent(rng, 40)\n        transaction = report - timedelta(days=int(rng.integers(10, 45)))\n        rows.append({\n            "Representative": rep,\n            "Party": "D" if _REPS.index(rep) % 2 == 0 else "R",\n            "House": rng.choice(["Representatives", "Senate"], p=[0.75, 0.25]),\n            "Ticker": ticker, "Transaction": txn, "Range": rng.choice(_RANGES),\n            "TransactionDate": transaction, "ReportDate": report,\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_insiders(seed=43, n=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.4 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Name": f"Insider {rng.integers(1000, 9999)}",\n            "Title": rng.choice(_TITLES), "Date": _recent(rng, 90),\n            "TransactionCode": rng.choice(["P", "S"], p=[0.6, 0.4]),\n            "Shares": int(rng.integers(500, 50000)),\n            "PricePerShare": round(float(rng.uniform(20, 400)), 2),\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_gov_contracts(seed=44, n=80):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.5 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, 90),\n            "Amount": int(rng.integers(50_000, 500_000_000)),\n            "Agency": rng.choice(_AGENCIES), "Description": "Procurement contract award",\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_lobbying(seed=45, n=140):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, 360),\n            "Amount": int(rng.integers(10_000, 5_000_000)),\n            "Client": f"{ticker} Inc.", "Issue": rng.choice(_ISSUES),\n        })\n    return pd.DataFrame(rows)\n')
open("src/quant_research/ingestion/client.py", "w").write('"""A unified client for Quiver Quantitative data.\n\nThe QuiverClient exposes one method per dataset. Each method fetches raw data\n(from the live API when a token is supplied, otherwise from the mock generator)\nand returns it normalized into a consistent internal schema. The rest of the\nproject depends only on that internal schema, so nothing downstream needs to\nknow or care whether the data came from the live API or the mock source.\n"""\nimport re\nimport pandas as pd\n\nfrom . import mock_data\n\n\ndef _parse_range(text):\n    """Turn a Quiver amount range like \'$15,001 - $50,000\' into (low, high)."""\n    nums = re.findall(r"[\\d,]+", str(text))\n    vals = [int(n.replace(",", "")) for n in nums if n.replace(",", "").isdigit()]\n    if len(vals) >= 2:\n        return vals[0], vals[1]\n    if len(vals) == 1:\n        return vals[0], vals[0]\n    return 0, 0\n\n\nclass QuiverClient:\n    def __init__(self, token=None, mock=False):\n        # No token means we cannot reach the live API, so we fall back to mock.\n        self.mock = mock or token is None\n        self._api = None\n        if not self.mock:\n            import quiverquant            # imported lazily; unused in mock mode\n            self._api = quiverquant.quiver(token)\n\n    # ---- Congressional trades -------------------------------------------\n    def congress_trades(self):\n        raw = (mock_data.mock_congress_trading() if self.mock\n               else self._api.congress_trading())\n        return self._normalize_congress(raw)\n\n    def _normalize_congress(self, df):\n        df = df.copy()\n        lows, highs = zip(*df["Range"].map(_parse_range)) if len(df) else ([], [])\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["TransactionDate"]),\n            "report_date": pd.to_datetime(df["ReportDate"]),\n            "representative": df["Representative"],\n            "party": df["Party"],\n            "chamber": df["House"],\n            "transaction_type": df["Transaction"],\n            "amount_min": list(lows),\n            "amount_max": list(highs),\n        })\n        return out\n\n    # ---- Insider transactions -------------------------------------------\n    def insider_trades(self):\n        raw = (mock_data.mock_insiders() if self.mock\n               else self._api.insiders())\n        return self._normalize_insiders(raw)\n\n    def _normalize_insiders(self, df):\n        df = df.copy()\n        code_map = {"P": "Purchase", "S": "Sale"}\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["Date"]),\n            "insider_name": df["Name"],\n            "title": df["Title"],\n            "transaction_type": df["TransactionCode"].map(code_map).fillna("Other"),\n            "shares": df["Shares"].astype(int),\n            "price_per_share": df["PricePerShare"].astype(float),\n        })\n        out["value"] = (out["shares"] * out["price_per_share"]).round(2)\n        return out\n\n    # ---- Government contracts --------------------------------------------\n    def gov_contracts(self):\n        raw = (mock_data.mock_gov_contracts() if self.mock\n               else self._api.gov_contracts())\n        return self._normalize_gov(raw)\n\n    def _normalize_gov(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "award_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "agency": df["Agency"],\n        })\n        return out\n\n    # ---- Lobbying --------------------------------------------------------\n    def lobbying(self):\n        raw = (mock_data.mock_lobbying() if self.mock\n               else self._api.lobbying())\n        return self._normalize_lobbying(raw)\n\n    def _normalize_lobbying(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "filing_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "client": df["Client"],\n            "issue": df["Issue"],\n        })\n        return out\n')
print("wrote ingestion layer")

wrote ingestion layer


In [5]:
open("src/quant_research/signals/base.py", "w").write('"""The signal contract every signal implements.\n\nA signal turns raw alternative data into a per-ticker score on a 0-100 scale, so\nthat scores from different datasets are directly comparable and can be blended\ninto a composite. Keeping every signal behind one small interface means the\nbacktester and the composite scorer depend on the contract, never on the details\nof any single dataset.\n"""\nfrom abc import ABC, abstractmethod\nfrom datetime import date\nimport pandas as pd\n\n\nclass BaseSignal(ABC):\n    name: str = "base"\n    description: str = ""\n\n    @abstractmethod\n    def compute(self, as_of: date | None = None) -> pd.DataFrame:\n        """Return a DataFrame indexed by ticker with a \'score\' column in [0, 100]."""\n        ...\n\n    @staticmethod\n    def _scale_0_100(s: pd.Series) -> pd.Series:\n        """Min-max a series onto [0, 100]; a flat series maps to zeros."""\n        if s.empty or s.max() == s.min():\n            return pd.Series(0.0, index=s.index)\n        return (s - s.min()) / (s.max() - s.min()) * 100.0\n')
open("src/quant_research/signals/congress.py", "w").write('"""Congressional cluster-buy signal.\n\nThe score rewards securities that several members of Congress are buying at once,\nwith each buy weighted by how informed it is likely to be. Features encode testable\nhypotheses; the backtest layer calibrates their weights.\n\nFeatures (each normalized across the cross-section, then weighted):\n  cluster              - distinct members buying the same name\n  size_vs_networth     - trade size relative to the member\'s net worth (conviction\n                         relative to means, controlling for the wealth confound)\n  committee_alignment  - buys by members whose committee oversees the ticker\'s sector\n  recency              - disclosures weighted toward the present\n  bipartisan           - a bonus when both parties are buying the same name\n\nMember net worth and committee membership come from the enrichment layer, which\nresolves a real implementation on live data and a synthetic one in mock mode.\n\nSignals are keyed on the DISCLOSURE (report) date, never the trade date, because a\ntrade is only actionable once it is public — which is what keeps the downstream\nbacktest free of look-ahead.\n"""\nfrom datetime import date\nimport numpy as np\nimport pandas as pd\n\nfrom .base import BaseSignal\nfrom ..enrichment.mock import MockEnrichment\nfrom ..enrichment.live import LiveEnrichment\n\nDEFAULT_WEIGHTS = {\n    "cluster": 0.30,\n    "size_vs_networth": 0.25,\n    "committee_alignment": 0.25,\n    "recency": 0.15,\n    "bipartisan": 0.05,\n}\n\n\nclass CongressSignal(BaseSignal):\n    name = "congress"\n    description = ("Clustered congressional purchases, weighted by conviction "\n                   "relative to net worth and by committee-sector alignment.")\n\n    def __init__(self, client, lookback_days=30, weights=None, enrichment=None):\n        self.client = client\n        self.lookback_days = lookback_days\n        self.weights = weights or dict(DEFAULT_WEIGHTS)\n        self.enrichment = enrichment or (\n            MockEnrichment() if getattr(client, "mock", False) else LiveEnrichment())\n\n    def compute(self, as_of=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        start = as_of - pd.Timedelta(days=self.lookback_days)\n\n        trades = self.client.congress_trades()\n        buys = trades[(trades.transaction_type == "Purchase")\n                      & (trades.report_date > start)\n                      & (trades.report_date <= as_of)].copy()\n        if buys.empty:\n            return pd.DataFrame(columns=["score"])\n\n        buys["mid_amount"] = (buys.amount_min + buys.amount_max) / 2.0\n\n        nw = buys.representative.map(self.enrichment.net_worth)\n        if nw.notna().any():\n            # conviction relative to means; unknown members fall back to the median\n            buys["conviction"] = buys.mid_amount / nw.fillna(nw.dropna().median())\n        else:\n            buys["conviction"] = buys.mid_amount      # no net worth available -> raw size\n\n        buys["days_ago"] = (as_of - buys.report_date).dt.days\n        buys["recency_w"] = np.select(\n            [buys.days_ago <= 7, buys.days_ago <= 14], [1.0, 0.6], default=0.3)\n        buys["aligned"] = [int(self.enrichment.is_aligned(r, t))\n                           for r, t in zip(buys.representative, buys.ticker)]\n\n        g = buys.groupby("ticker")\n        feat = pd.DataFrame({\n            "n_buys": g.size(),\n            "cluster": g.representative.nunique(),\n            "size_vs_networth": g.conviction.sum(),\n            "committee_alignment": g.aligned.sum(),\n            "recency": g.recency_w.sum(),\n            "bipartisan": g.party.apply(lambda p: int({"D", "R"}.issubset(set(p)))),\n        })\n\n        norm = {}\n        for f in ["cluster", "size_vs_networth", "committee_alignment", "recency"]:\n            col = feat[f].astype(float)\n            spread = col.max() - col.min()\n            norm[f] = (col - col.min()) / spread if spread > 0 else col * 0.0\n        norm["bipartisan"] = feat["bipartisan"].astype(float)\n        norm = pd.DataFrame(norm)\n\n        raw = sum(self.weights[f] * norm[f] for f in self.weights)\n        feat["score"] = self._scale_0_100(raw).round(1)\n        for f in norm.columns:\n            feat[f + "_n"] = norm[f].round(3)\n        return feat.sort_values("score", ascending=False)\n')
print("wrote signal layer")

wrote signal layer


In [6]:
open("reference_data/networth.csv", "w").write('bioguide,name,net_worth,as_of,source\nP000197,Nancy Pelosi,120000000,2024,approx public estimate\nT000278,Tommy Tuberville,6000000,2024,approx public estimate\nK000389,Ro Khanna,30000000,2024,approx public estimate\nM001157,Michael McCaul,100000000,2024,approx public estimate\nC001120,Dan Crenshaw,3000000,2024,approx public estimate\n')
open("pyproject.toml", "w").write('[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "quant-equity-research"\nversion = "0.1.0"\ndescription = "Equity research signals from alternative data"\nreadme = "README.md"\nrequires-python = ">=3.10"\ndependencies = [\n    "quiverquant",\n    "pandas",\n    "numpy",\n    "sqlalchemy",\n    "yfinance",\n    "plotly",\n    "scipy",\n    "pyyaml",\n]\n\n[project.optional-dependencies]\ndev = ["pytest"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n')
# the old flat reference module, if present from an earlier build, is superseded
import os
if os.path.exists("src/quant_research/reference.py"): os.remove("src/quant_research/reference.py")
print("wrote reference data and pyproject")

wrote reference data and pyproject


Install in editable mode and put the package on the session path. The added
`src` path entry lets the kernel import the freshly written modules — including the
new `enrichment` subpackage — without a restart.


In [7]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import quant_research
importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.1.0


## The enrichment layer

Net worth and committee membership are the two features the Quiver client cannot
supply, so the project sources them independently.

- **Committee membership** comes from the `@unitedstates/congress-legislators`
  project — an authoritative open dataset. A disclosed trade carries a member's
  name; the layer resolves that name to a bioguide id (handling middle names and
  suffixes), then to the member's current committees, and maps each committee to
  the sectors it oversees.
- **Net worth** comes from a curated reference table (`reference_data/networth.csv`),
  seeded with approximate public estimates and meant to be expanded from primary
  financial disclosures.
- **Ticker sector** comes from a static map, with a yfinance fallback for names not
  yet mapped.

Members the data cannot match, and tickers without a sector, degrade quietly — the
feature declines to fire rather than penalizing the name. Building `LiveEnrichment`
fetches the committee data once and caches it locally.


In [8]:
from quant_research.enrichment.live import LiveEnrichment

live = LiveEnrichment()   # one-time fetch of public committee data, cached to data/reference

import pandas as pd
demo = []
for name in ["Tommy Tuberville", "Dan Crenshaw", "Michael McCaul", "Ro Khanna"]:
    demo.append({
        "member": name,
        "bioguide": live.resolve(name),
        "net_worth": live.net_worth(name),
        "committee_sectors": ", ".join(sorted(live.committee_sectors(name))) or "—",
    })
pd.DataFrame(demo)

,member,bioguide,net_worth,committee_sectors
0,Tommy Tuberville,T000278,6000000.0,"Consumer Staples, Defense, Health Care"
1,Dan Crenshaw,C001120,3000000.0,"Consumer Discretionary, Energy, Technology, Ut..."
2,Michael McCaul,M001157,100000000.0,Defense
3,Ro Khanna,K000389,30000000.0,Defense


A quick alignment check confirms the join is doing real work: an Energy-committee
member aligns with an energy name, an Armed Services member with a defense name.


In [9]:
print("Crenshaw (Energy committee)  + XOM [Energy]   ->", live.is_aligned("Dan Crenshaw", "XOM"))
print("Tuberville (Armed Services)  + LMT [Defense]  ->", live.is_aligned("Tommy Tuberville", "LMT"))
print("Crenshaw                     + JPM [Financials]->", live.is_aligned("Dan Crenshaw", "JPM"))

Crenshaw (Energy committee)  + XOM [Energy]   -> True
Tuberville (Armed Services)  + LMT [Defense]  -> True
Crenshaw                     + JPM [Financials]-> False


## Construct the signal

We compute the signal as of today, passing the **live enrichment** so the net-worth
and committee features run on real data — even on mock trades, since the mock member
names are real. `compute` returns a DataFrame indexed by ticker, sorted by score,
carrying both the final `score` and the features behind it, so every score is fully
explainable.


In [10]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = CongressSignal(client, lookback_days=30, enrichment=live)

scores = signal.compute()
print(f"{len(scores)} tickers scored | trades: {'live' if USE_LIVE else 'mock'} | enrichment: live")
scores[["score", "n_buys", "cluster", "committee_alignment", "bipartisan"]].head(12)

https://api.quiverquant.com/beta/live/congresstrading
82 tickers scored | trades: live | enrichment: live


,score,n_buys,cluster,committee_alignment,bipartisan
ticker,,,,,
MSFT,100.0,4,3,0,1
AAPL,58.4,3,3,0,1
ENTG,50.7,3,1,3,0
AMD,37.2,5,2,0,0
T,36.4,4,2,0,0
PH,29.4,2,1,2,0
FN,28.8,2,2,0,0
HUBB,24.1,5,1,0,0
TDG,23.4,5,1,0,0


### Reading the output

`score` is the final 0–100 conviction reading. Beside it sit the raw features, and
the `*_n` columns below are those features normalized across the cross-section — the
actual inputs to the weighted score.


In [11]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,cluster_n,size_vs_networth_n,committee_alignment_n,recency_n,bipartisan_n
ticker,,,,,,
MSFT,100.0,1.0,1.000,0.000,0.468,1.0
AAPL,58.4,1.0,0.013,0.000,0.255,1.0
ENTG,50.7,0.0,0.013,1.000,0.574,0.0
AMD,37.2,0.5,0.027,0.000,0.617,0.0
T,36.4,0.5,0.096,0.000,0.468,0.0
PH,29.4,0.0,0.007,0.667,0.191,0.0
FN,28.8,0.5,0.007,0.000,0.277,0.0
HUBB,24.1,0.0,0.047,0.000,1.000,0.0


### Why the leaders score where they do

A score is only useful if it is explainable. The chart shades each candidate by
committee alignment, so the contribution of that feature is visible at a glance,
alongside the cluster and conviction components.


In [12]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="committee_alignment_n", color_continuous_scale="Tealgrn",
             labels={"committee_alignment_n": "committee\nalignment"},
             title="Congress signal — top 15 (shading = committee alignment)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## The disclosure-lag discipline, in practice

The signal keys on the disclosure date rather than the trade date. Quantifying the
gap makes the look-ahead concern concrete: a backtest scoring these trades on their
*transaction* date would be acting on information not yet public.


In [13]:
raw = client.congress_trades()
lag = (raw.report_date - raw.transaction_date).dt.days
print("Disclosure lag (days) — min / mean / max:",
      int(lag.min()), "/", round(lag.mean(), 1), "/", int(lag.max()))
print("Share disclosed more than 30 days after the trade:", f"{(lag > 30).mean():.0%}")

https://api.quiverquant.com/beta/live/congresstrading
Disclosure lag (days) — min / mean / max: 1 / 23.0 / 336
Share disclosed more than 30 days after the trade: 17%


## What notebook 02 will settle

The feature weights here are **priors**, not conclusions. The next notebook runs an
event study: for each disclosure it measures forward returns at several horizons,
then buckets those returns by feature — large committee-aligned buys in one bucket,
small unaligned ones in another. A feature that genuinely separates winners from
losers earns its weight from the data; one that does not is dropped to guard against
overfitting.


## Commit

We commit the signal and enrichment layers and the curated reference table. The
author identity is set so the work is attributed to you.


In [14]:
!git config --global user.email "aballa1234@gmail.com"   # <-- edit
!git config --global user.name "Alex Balla"          # <-- edit

!git add -A
!git commit -m "Add congress signal and enrichment layer (committees, net worth, sectors)"
!git push

[main 60c6d62] Add congress signal and enrichment layer (committees, net worth, sectors)
 12 files changed, 410 insertions(+), 35 deletions(-)
 create mode 100644 data/reference/committee-membership-current.json
 create mode 100644 data/reference/committees-current.json
 create mode 100644 data/reference/legislators-current.json
 create mode 100644 reference_data/networth.csv
 create mode 100644 src/quant_research/enrichment/__init__.py
 create mode 100644 src/quant_research/enrichment/live.py
 create mode 100644 src/quant_research/enrichment/mock.py
 create mode 100644 src/quant_research/signals/base.py
 create mode 100644 src/quant_research/signals/congress.py
Enumerating objects: 31, done.
Counting objects: 100% (31/31), done.
Delta compression using up to 2 threads
Compressing objects: 100% (20/20), done.
Writing objects: 100% (22/22), 170.85 KiB | 2.59 MiB/s, done.
Total 22 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local ob

## Recap and next

The first signal is built and genuinely complete: a transparent congressional
cluster-buy score whose conviction feature normalizes against real net worth and
whose alignment feature draws on real committee assignments, all respecting
disclosure timing. The enrichment layer that powers it is reusable by every signal
that follows.

Next, the remaining Hobbyist-tier signals — government contracts, lobbying, and
off-exchange — adopt the same `BaseSignal` contract, and a weighted composite blends
them into one ranking. Then notebook `02` runs the event study that calibrates these
weights against real forward returns.
